In [1]:
import mtl, pandas as pd , os

#print(os.getcwd())
#os.chdir("c:/Users/Gugu/Documents/UAntwerpen/mosis-assignment-1")
CURRENT_DIR = os.getcwd()

PROJECT_ROOT = os.path.abspath(
    os.path.join(CURRENT_DIR, "..", "..")
)

os.chdir(PROJECT_ROOT)

print(os.getcwd())

/Users/zlata/Library/CloudStorage/OneDrive-Personal/UAntwerpen/1_semester/Modelling_of_Software_Intensive_Systems/mosis_1


In [2]:
data=["data/stl/gantry/gantry_system_kp1_ki0_kd10.csv",
     "data/stl/gantry/gantry_system_kp10_ki50_kd10.csv",
      "data/stl/gantry/gantry_system_kp50_ki0_kd100.csv",
       "data/stl/gantry/gantry_system_kp50_ki1_kd100.csv",
        "data/stl/gantry/gantry_system_kp50_ki10_kd100.csv",
         "data/stl/gantry/gantry_system_kp100_ki5_kd50.csv",
          "data/stl/gantry/gantry_system_kp200_ki0_kd400.csv",
           "data/stl/gantry/gantry_system_kp200_ki20_kd100.csv",
             "data/stl/gantry/gantry_system_kp400_ki2_kd50.csv",
             "data/stl/gantry/gantry_system_kp500_ki0_kd200.csv"]

In [3]:
traces=[]

for file in data:
    frame = pd.read_csv(file)
    traces.append({
        #angle_within_half: the absolute angle is at most 0.5 rad.
        "angle_within_half": list(zip(frame["time"], (frame["gantry.theta"].abs()<=0.5)) ),
        #angle_within_limit: the absolute angle is at most 1.5 rad.
        "angle_within_limit": list(zip(frame["time"], (frame["gantry.theta"].abs()<=1.5)) ),
        #reaches_by_8s: the position reaches at least 9.5 within the first 8 s.
        "reaches_by_8s": list(zip(frame["time"], (frame["gantry.x"].abs()>=9.5)) ),
        #settled: from time 10 through the end, the position stays within 0.5 of the setpoint.
        "settled": list(zip(frame["time"], (frame["gantry.x"].abs()>=9.5) & (frame["gantry.x"].abs()<=10.5)) ),
        #no_overshoot: the position is always at most 10.5..
        "no_overshoot": list(zip(frame["time"], (frame["gantry.x"].abs()<=10.5)) ) 

    })

angle_within_half = mtl.parse("G[0, 20.1] (angle_within_half)")
angle_within_limit = mtl.parse("G[0, 20.1] (angle_within_limit)")
reaches_by_8s = mtl.parse("F[0,8.1] (reaches_by_8s)")
settled= mtl.parse("G[10,20.1 ] (settled)")
no_overshoot = mtl.parse("G[0,20.1 ] (no_overshoot)")

properties = {
    "angle_within_half": angle_within_half,
    "angle_within_limit": angle_within_limit,
    "reaches_by_8s": reaches_by_8s,
    "settled": settled,
    "no_overshoot": no_overshoot

}


results = []

for i, trace in enumerate(traces):

    row = {
        "trace": os.path.basename(data[i])
    }

    for name, phi in properties.items():

        row[name] = phi( trace,time=0,quantitative=False,dt=1)

    results.append(row)
    

In [4]:
results_df = pd.DataFrame(results)

print(results_df)
results_df= results_df.replace({True: "true", False: "false"})

results_df.to_csv("results/stl/gantry_system_stl.csv", index=False)

                                trace  angle_within_half  angle_within_limit  \
0      gantry_system_kp1_ki0_kd10.csv               True                True   
1    gantry_system_kp10_ki50_kd10.csv              False               False   
2    gantry_system_kp50_ki0_kd100.csv              False                True   
3    gantry_system_kp50_ki1_kd100.csv              False                True   
4   gantry_system_kp50_ki10_kd100.csv              False                True   
5    gantry_system_kp100_ki5_kd50.csv              False               False   
6   gantry_system_kp200_ki0_kd400.csv              False                True   
7  gantry_system_kp200_ki20_kd100.csv              False               False   
8    gantry_system_kp400_ki2_kd50.csv              False               False   
9   gantry_system_kp500_ki0_kd200.csv              False               False   

   reaches_by_8s  settled  no_overshoot  
0          False    False          True  
1           True    False         F